# Demo: Part 2 - base SD 1.5 vs our fine-tuned LoRA
Generates the graders' prompts with the **base model** and our **final LoRA (run 1, rank 4)** from Hugging Face,
with the same sampler as the evaluation (DPM-Solver++, 30 steps, guidance 7.5, seeds 0 and 1). Safety checker on.

**Before the demo:** Colab, Runtime → **T4 GPU**. Secret: `GITHUB_TOKEN`. Run all cells once (first run downloads
SD 1.5, about 3 min).

In [ ]:
# Cell 1 - run information: clone the graded tag, then print commit, versions, GPU and seed
import os, sys, subprocess, platform, random, datetime
import numpy as np
TAG, REPO, WORK = "p2-v1.0", "sakotiya/data266-lab2-team32", "/content/data266-lab2-team32"
from google.colab import userdata
if not os.path.isdir(WORK):
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "-q", "-b", TAG, f"https://{token}@github.com/{REPO}.git", WORK], check=True)
os.chdir(WORK); sys.path.insert(0, WORK)
SEED = 266; random.seed(SEED); np.random.seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print({"part": 2, "tag": TAG, "commit": commit, "python": platform.python_version(),
       "gpu": gpu or "cpu", "seed": SEED, "time": datetime.datetime.now().isoformat(timespec="minutes")})

In [ ]:
!pip install -q diffusers==0.42.1 transformers==5.22.1 peft==0.21.2
!pip uninstall -y -q torchao     # Colab's torchao 0.10 makes peft refuse to load LoRAs
import torch, yaml, diffusers, peft
print({"torch": torch.__version__, "diffusers": diffusers.__version__, "peft": peft.__version__,
       "gpu": torch.cuda.get_device_name(0)})

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
cfg = yaml.safe_load(open("part2_diffusion/configs/config.yaml"))
GEN = cfg["generate"]
LORA = "sakotiya/data266-team32-naruto-run1-lora-r4"     # final model (run 1, rank 4)
pipe = StableDiffusionPipeline.from_pretrained(cfg["base_model"], torch_dtype=torch.float16, variant="fp16")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda"); pipe.set_progress_bar_config(disable=True)
pipe.load_lora_weights(LORA, adapter_name="r4")
print("loaded", cfg["base_model"], "+ LoRA", LORA, "| sampler:", GEN)

In [ ]:
import time, matplotlib.pyplot as plt

def generate(prompt, use_lora, seed):
    (pipe.enable_lora if use_lora else pipe.disable_lora)()
    out = pipe(prompt, num_inference_steps=GEN["steps"], guidance_scale=GEN["guidance_scale"],
               generator=torch.Generator("cuda").manual_seed(seed))
    return out.images[0], bool(out.nsfw_content_detected and out.nsfw_content_detected[0])

def compare(prompt, seeds=tuple(GEN["seeds"])):
    """Base vs LoRA r4 for each seed, side by side."""
    t = time.perf_counter()
    fig, axes = plt.subplots(len(seeds), 2, figsize=(9, 4.6 * len(seeds)), squeeze=False)
    for row, seed in enumerate(seeds):
        for col, (name, lora) in enumerate([("base SD 1.5", False), ("LoRA r4 (ours)", True)]):
            img, nsfw = generate(prompt, lora, seed)
            axes[row, col].imshow(img); axes[row, col].axis("off")
            axes[row, col].set_title(f"{name}, seed {seed}" + ("  [flagged by safety checker]" if nsfw else ""), fontsize=10)
    fig.suptitle(prompt, fontsize=11); plt.tight_layout(); plt.show()
    print(f"{2 * len(seeds)} images in {time.perf_counter() - t:.1f} s")

## Warm-up: one of our fixed prompts

In [ ]:
import pandas as pd
prompts = pd.read_csv("part2_diffusion/eval/prompts.csv")
display(prompts)
compare(prompts.loc[prompts.prompt_id == "OD01", "prompt"].item())

## Graders' prompts

In [ ]:
compare("TYPE THE GRADER'S FIRST PROMPT HERE")

In [ ]:
compare("TYPE THE GRADER'S SECOND PROMPT HERE")